In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import folium
import json
import struct
import branca.colormap as cm
import math

# =========================
# COORDINATE TRANSFORMATION
# =========================

def stateplane_to_latlon(x_feet, y_feet):
    """
    Convert NAD83 State Plane Florida North (feet) to Lat/Lon
    Uses Lambert Conformal Conic projection parameters
    """
    # Convert feet to meters
    feet_to_meters = 0.3048006096012192
    x_m = x_feet * feet_to_meters
    y_m = y_feet * feet_to_meters

    # Projection parameters for Florida North State Plane
    false_easting = 1968500.0 * feet_to_meters
    false_northing = 0.0
    central_meridian = -84.5
    lat_origin = 29.0
    std_parallel_1 = 29.58333333333333
    std_parallel_2 = 30.75

    # Semi-major axis for GRS 1980
    a = 6378137.0
    e = 0.08181919084262157  # eccentricity

    # Convert to radians
    phi0 = math.radians(lat_origin)
    phi1 = math.radians(std_parallel_1)
    phi2 = math.radians(std_parallel_2)
    lambda0 = math.radians(central_meridian)

    # Lambert Conformal Conic formulas
    def m_calc(phi):
        return math.cos(phi) / math.sqrt(1 - e**2 * math.sin(phi)**2)

    def t_calc(phi):
        return math.tan(math.pi/4 - phi/2) / ((1 - e*math.sin(phi))/(1 + e*math.sin(phi)))**(e/2)

    m1 = m_calc(phi1)
    m2 = m_calc(phi2)
    t0 = t_calc(phi0)
    t1 = t_calc(phi1)
    t2 = t_calc(phi2)

    n = (math.log(m1) - math.log(m2)) / (math.log(t1) - math.log(t2))
    F = m1 / (n * t1**n)
    rho0 = a * F * t0**n

    # Inverse projection
    x_adj = x_m - false_easting
    y_adj = y_m - false_northing

    rho = math.sqrt(x_adj**2 + (rho0 - y_adj)**2) * (-1 if n < 0 else 1)
    theta = math.atan2(x_adj, rho0 - y_adj)

    t = (rho / (a * F))**(1/n)

    # Iterate to find latitude
    phi = math.pi/2 - 2 * math.atan(t)
    for _ in range(10):  # iterations
        phi_new = math.pi/2 - 2 * math.atan(t * ((1 - e*math.sin(phi))/(1 + e*math.sin(phi)))**(e/2))
        if abs(phi_new - phi) < 1e-10:
            break
        phi = phi_new

    lambda_val = theta / n + lambda0

    lat = math.degrees(phi)
    lon = math.degrees(lambda_val)

    return lon, lat


def albers_to_latlon(x_m, y_m):
    """
    Convert NAD83 HARN Florida GDL Albers (meters) to Lat/Lon
    Uses Albers Equal Area Conic projection parameters
    """
    # Projection parameters for Florida GDL Albers
    false_easting = 400000.0
    false_northing = 0.0
    central_meridian = -84.0
    lat_origin = 24.0
    std_parallel_1 = 24.0
    std_parallel_2 = 31.5

    # Semi-major axis for GRS 1980
    a = 6378137.0
    e = 0.08181919084262157  # eccentricity
    e_sq = e * e

    # Convert to radians
    phi0 = math.radians(lat_origin)
    phi1 = math.radians(std_parallel_1)
    phi2 = math.radians(std_parallel_2)
    lambda0 = math.radians(central_meridian)

    # Helper function for q
    def q_calc(phi):
        sin_phi = math.sin(phi)
        return (1 - e_sq) * (
            sin_phi / (1 - e_sq * sin_phi**2) -
            (1 / (2 * e)) * math.log((1 - e * sin_phi) / (1 + e * sin_phi))
        )

    # Helper function for m
    def m_calc(phi):
        return math.cos(phi) / math.sqrt(1 - e_sq * math.sin(phi)**2)

    # Calculate constants
    m1 = m_calc(phi1)
    m2 = m_calc(phi2)
    q0 = q_calc(phi0)
    q1 = q_calc(phi1)
    q2 = q_calc(phi2)

    n = (m1**2 - m2**2) / (q2 - q1)
    C = m1**2 + n * q1
    rho0 = a * math.sqrt(C - n * q0) / n

    # Inverse projection
    x_adj = x_m - false_easting
    y_adj = y_m - false_northing

    rho = math.sqrt(x_adj**2 + (rho0 - y_adj)**2)
    if n < 0:
        rho = -rho

    theta = math.atan2(x_adj, rho0 - y_adj)
    q = (C - (rho * n / a)**2) / n

    # Iterative solution for latitude
    phi = math.asin(q / 2)
    for _ in range(10):
        sin_phi = math.sin(phi)
        term = (1 - e_sq * sin_phi**2)**2 / (2 * math.cos(phi))
        delta = term * (
            q / (1 - e_sq) -
            sin_phi / (1 - e_sq * sin_phi**2) +
            (1 / (2 * e)) * math.log((1 - e * sin_phi) / (1 + e * sin_phi))
        )
        phi += delta
        if abs(delta) < 1e-10:
            break

    lambda_val = lambda0 + theta / n

    lat = math.degrees(phi)
    lon = math.degrees(lambda_val)

    return lon, lat


# =========================
# SHAPEFILE READER FUNCTIONS
# =========================

def read_dbf(filename):
    """Read DBF file and return records as list of dicts"""
    with open(filename, 'rb') as f:
        # Read header
        header = f.read(32)
        num_records = struct.unpack('<I', header[4:8])[0]
        header_length = struct.unpack('<H', header[8:10])[0]
        record_length = struct.unpack('<H', header[10:12])[0]

        # Read field descriptors
        fields = []
        f.seek(32)
        while True:
            field_info = f.read(32)
            if field_info[0] == 0x0D:  # End of field descriptors
                break
            field_name = field_info[:11].split(b'\x00')[0].decode('ascii')
            field_type = chr(field_info[11])
            field_length = field_info[16]
            fields.append((field_name, field_type, field_length))

        # Read records
        f.seek(header_length)
        records = []
        for _ in range(num_records):
            record = {}
            deleted = f.read(1)
            if deleted == b'*':
                f.read(record_length - 1)
                continue

            for field_name, field_type, field_length in fields:
                value = f.read(field_length).strip()
                if field_type == 'C':  # Character
                    record[field_name] = value.decode('ascii', errors='ignore').strip()
                elif field_type == 'N':  # Numeric
                    try:
                        if b'.' in value:
                            record[field_name] = float(value)
                        else:
                            record[field_name] = int(value)
                    except:
                        record[field_name] = 0
                elif field_type == 'F':  # Float
                    try:
                        record[field_name] = float(value) if value else 0.0
                    except:
                        record[field_name] = 0.0
                elif field_type == 'D':  # Date
                    record[field_name] = value.decode('ascii', errors='ignore').strip()
                else:
                    record[field_name] = value.decode('ascii', errors='ignore').strip()
            records.append(record)

    return records, fields


def read_shp(filename):
    """Read SHP file and return geometries"""
    with open(filename, 'rb') as f:
        # Read header
        header = f.read(100)

        geometries = []
        while True:
            # Read record header
            record_header = f.read(8)
            if not record_header or len(record_header) < 8:
                break

            record_number = struct.unpack('>I', record_header[0:4])[0]
            content_length = struct.unpack('>I', record_header[4:8])[0] * 2

            # Read shape type
            shape_type = struct.unpack('<I', f.read(4))[0]

            if shape_type == 0:  # Null shape
                geometries.append(None)
                continue

            if shape_type == 1:  # Point
                x, y = struct.unpack('<2d', f.read(16))
                geometries.append({
                    "type": "Point",
                    "coordinates": [x, y]
                })
            elif shape_type == 3 or shape_type == 23:  # PolyLine or PolyLineM
                # Read bounding box
                bbox = struct.unpack('<4d', f.read(32))

                # Read parts and points
                num_parts = struct.unpack('<I', f.read(4))[0]
                num_points = struct.unpack('<I', f.read(4))[0]

                # Read part indices
                parts = [struct.unpack('<I', f.read(4))[0] for _ in range(num_parts)]

                # Read points
                points = []
                for _ in range(num_points):
                    x, y = struct.unpack('<2d', f.read(16))
                    points.append([x, y])

                # Skip M values if PolyLineM (shape_type == 23)
                if shape_type == 23:
                    # Read M range
                    m_range = struct.unpack('<2d', f.read(16))
                    # Read M values
                    for _ in range(num_points):
                        f.read(8)  # Skip M value

                # Organize into line strings
                lines = []
                for i in range(num_parts):
                    start = parts[i]
                    end = parts[i + 1] if i + 1 < num_parts else num_points
                    line = points[start:end]
                    lines.append(line)

                # Create MultiLineString if multiple parts, otherwise LineString
                if len(lines) == 1:
                    geometries.append({
                        "type": "LineString",
                        "coordinates": lines[0],
                        "needs_transform": True  # Flag for State Plane coordinates
                    })
                else:
                    geometries.append({
                        "type": "MultiLineString",
                        "coordinates": lines,
                        "needs_transform": True  # Flag for State Plane coordinates
                    })
            elif shape_type == 5:  # Polygon
                # Read bounding box
                bbox = struct.unpack('<4d', f.read(32))

                # Read parts and points
                num_parts = struct.unpack('<I', f.read(4))[0]
                num_points = struct.unpack('<I', f.read(4))[0]

                # Read part indices
                parts = [struct.unpack('<I', f.read(4))[0] for _ in range(num_parts)]

                # Read points
                points = []
                for _ in range(num_points):
                    x, y = struct.unpack('<2d', f.read(16))
                    points.append([x, y])

                # Organize into rings
                rings = []
                for i in range(num_parts):
                    start = parts[i]
                    end = parts[i + 1] if i + 1 < num_parts else num_points
                    ring = points[start:end]
                    rings.append(ring)

                # Create polygon - flag if coordinates are very large (projected)
                needs_transform = any(abs(coord) > 360 for ring in rings for point in ring for coord in point)

                geometries.append({
                    "type": "Polygon",
                    "coordinates": rings,
                    "needs_transform": needs_transform
                })
            else:
                # Skip unknown shape types
                f.read(content_length - 4)
                geometries.append(None)

    return geometries


def convert_tractce_to_csv_format(tractce):
    """
    Convert shapefile TRACTCE format to CSV format.
    Shapefile uses 6-digit codes like '000301' (= 3.01), '001100' (= 11.00), '110800' (= 1108.00)
    CSV uses formats like '3.01', '11', '1108'
    """
    if not tractce:
        return None

    # Convert to string and pad if necessary
    tractce_str = str(tractce).zfill(6)

    # Split into whole and decimal parts (first 4 digits . last 2 digits)
    whole = tractce_str[:4]
    decimal = tractce_str[4:6]

    # Remove leading zeros from whole part
    whole_int = int(whole)

    # Check if decimal part is non-zero
    if decimal != '00':
        # Return with decimal
        return f"{whole_int}.{decimal}"
    else:
        # Return without decimal
        return str(whole_int)


def parse_wkt_multipolygon(wkt_string):
    """
    Parse WKT MULTIPOLYGON format to GeoJSON
    """
    import re

    # Handle MULTIPOLYGON with multiple polygons
    polygon_pattern = r'\(\(([-\d\s.,]+?)\)\)'
    polygons = re.findall(polygon_pattern, wkt_string)

    all_polygon_coords = []

    for polygon_str in polygons:
        # Split coordinate pairs
        coord_pairs = polygon_str.split(', ')
        coords = []
        for pair in coord_pairs:
            parts = pair.strip().split(' ')
            if len(parts) == 2:
                lon, lat = parts
                # GeoJSON uses [lon, lat] format
                coords.append([float(lon), float(lat)])

        if coords:
            all_polygon_coords.append([coords])  # Wrap in extra list for polygon rings

    return {
        "type": "MultiPolygon",
        "coordinates": all_polygon_coords
    }


# =========================
# FILE PATHS - GOOGLE DRIVE
# =========================

# Files in your Google Drive "capstone" folder
BUS_STOPS_SHP = "/content/drive/MyDrive/capstone/Random.shp"
BUS_STOPS_DBF = "/content/drive/MyDrive/capstone/Random.dbf"
BUS_ROUTES_SHP = "/content/drive/MyDrive/capstone/Spring2026_Weekday.shp"
BUS_ROUTES_DBF = "/content/drive/MyDrive/capstone/Spring2026_Weekday.dbf"
CITY_LIMITS_SHP = "/content/drive/MyDrive/capstone/par_citylm_2021.shp"
CITY_LIMITS_DBF = "/content/drive/MyDrive/capstone/par_citylm_2021.dbf"
SHAPEFILE_SHP = "/content/drive/MyDrive/capstone/tl_2020_12_tract.shp"
SHAPEFILE_DBF = "/content/drive/MyDrive/capstone/tl_2020_12_tract.dbf"
CENSUS_DATA_FILE = "/content/drive/MyDrive/capstone/c_t_sheet_1.csv"
LIBRARY_FILE = "/content/drive/MyDrive/capstone/aclib.csv"


# =========================
# 1. LOAD DATA
# =========================

print("Loading data...")

# Bus stop data from shapefile
try:
    bus_geometries = read_shp(BUS_STOPS_SHP)
    bus_records, bus_fields = read_dbf(BUS_STOPS_DBF)

    # Convert to a more usable format
    stops = []
    for i, record in enumerate(bus_records):
        if bus_geometries[i] and bus_geometries[i]['type'] == 'Point':
            coords = bus_geometries[i]['coordinates']
            stops.append({
                'BSID': record.get('BSID', ''),
                'stop_name': record.get('STOP_NAME', ''),
                'stop_desc': record.get('DESCRIPTIO', ''),
                'Latitude': coords[1],  # Point coordinates are [lon, lat]
                'Longitude': coords[0],
                'STREET': record.get('STREET', ''),
                'ACTIVE_INA': record.get('ACTIVE_INA', ''),
                'NO_BENCHES': record.get('NO_BENCHES', 0),
                'NO_SHELTER': record.get('NO_SHELTER', 0),
                'NO_BIKERAC': record.get('NO_BIKERAC', 0),
                'LANDING_PA': record.get('LANDING_PA', ''),
                'WAITING_PA': record.get('WAITING_PA', ''),
            })

    print(f"✓ Loaded {len(stops)} bus stops from shapefile")
except FileNotFoundError as e:
    print(f"ERROR: Could not find bus stops shapefile: {e}")
    print("Please make sure Random.shp and Random.dbf are in your 'capstone' folder")
    stops = []

# Bus routes data from shapefile
try:
    route_geometries = read_shp(BUS_ROUTES_SHP)
    route_records, route_fields = read_dbf(BUS_ROUTES_DBF)

    # Convert to a more usable format
    routes = []
    for i, record in enumerate(route_records):
        if route_geometries[i]:
            geom = route_geometries[i]

            # Transform coordinates from State Plane to Lat/Lon if needed
            if geom.get('needs_transform', False):
                if geom['type'] == 'LineString':
                    # Transform each coordinate
                    transformed_coords = []
                    for x, y in geom['coordinates']:
                        lon, lat = stateplane_to_latlon(x, y)
                        transformed_coords.append([lon, lat])
                    geom['coordinates'] = transformed_coords
                elif geom['type'] == 'MultiLineString':
                    # Transform each line in the multilinestring
                    transformed_lines = []
                    for line in geom['coordinates']:
                        transformed_coords = []
                        for x, y in line:
                            lon, lat = stateplane_to_latlon(x, y)
                            transformed_coords.append([lon, lat])
                        transformed_lines.append(transformed_coords)
                    geom['coordinates'] = transformed_lines

                # Remove the flag
                del geom['needs_transform']

            routes.append({
                'geometry': geom,
                'route_id': record.get('route_id', ''),
                'route_shor': record.get('route_shor', ''),
                'route_long': record.get('route_long', ''),
                'shape_id': record.get('shape_id', ''),
                'start_time': record.get('start_time', ''),
                'end_time': record.get('end_time', ''),
                'Colors': record.get('Colors', ''),
                'State': record.get('State', 0),
                'RouteType': record.get('RouteType', 0),
                'length_mi': record.get('length_mi', 0),
                'AM_freq': record.get('AM_freq', 0),
                'Midday_fre': record.get('Midday_fre', 0),
                'PM_freq': record.get('PM_freq', 0),
            })

    print(f"✓ Loaded {len(routes)} bus routes from shapefile")

    # Get unique route numbers
    unique_routes = set(r['route_shor'] for r in routes if r['route_shor'])
    print(f"  Unique routes: {sorted(unique_routes)}")

except FileNotFoundError as e:
    print(f"ERROR: Could not find bus routes shapefile: {e}")
    print("Please make sure Spring2026_Weekday.shp and .dbf are in your 'capstone' folder")
    routes = []

# Census tract FB data
try:
    census_data = pd.read_csv(CENSUS_DATA_FILE)
    # Remove commas from numeric columns and convert to int
    census_data['FB'] = census_data['FB'].astype(str).str.replace(',', '').astype(int)
    census_data['Total'] = census_data['Total'].astype(str).str.replace(',', '').astype(int)
    # Calculate FB percentage
    census_data['FB_pct'] = (census_data['FB'] / census_data['Total'] * 100).round(2)
    # Extract tract code (remove "CT" prefix)
    census_data['TRACTCE'] = census_data['Census Tract'].str.replace('CT', '')
    print(f"✓ Loaded {len(census_data)} census tracts with FB data")
except FileNotFoundError:
    print(f"ERROR: Could not find '{CENSUS_DATA_FILE}'")
    census_data = None

# Gainesville city boundary from shapefile
try:
    city_geometries = read_shp(CITY_LIMITS_SHP)
    city_records, city_fields = read_dbf(CITY_LIMITS_DBF)

    # Find Gainesville
    gainesville_geom = None
    gainesville_acres = None

    for i, record in enumerate(city_records):
        if record.get('NAME', '').upper() == 'GAINESVILLE':
            if city_geometries[i]:
                geom = city_geometries[i]

                # Transform coordinates if needed (Albers projection)
                if geom.get('needs_transform', False):
                    transformed_rings = []
                    for ring in geom['coordinates']:
                        transformed_coords = []
                        for x, y in ring:
                            lon, lat = albers_to_latlon(x, y)
                            transformed_coords.append([lon, lat])
                        transformed_rings.append(transformed_coords)
                    geom['coordinates'] = transformed_rings
                    del geom['needs_transform']

                gainesville_geom = geom
                gainesville_acres = record.get('ACRES', 0)
                print(f"✓ Loaded Gainesville city limits ({gainesville_acres:.1f} acres = {gainesville_acres/640:.1f} sq mi)")
                break

    if not gainesville_geom:
        print("WARNING: Gainesville not found in city limits shapefile")

except FileNotFoundError as e:
    print(f"ERROR: Could not find city limits shapefile: {e}")
    gainesville_geom = None
    gainesville_acres = None

# Library locations
try:
    library_data = pd.read_csv(LIBRARY_FILE)
    libraries = []
    for _, row in library_data.iterrows():
        libraries.append({
            'Branch': row['Branch'],
            'Latitude': row['Latitude'],
            'Longitude': row['Longitude']
        })
    print(f"✓ Loaded {len(libraries)} library locations")
except FileNotFoundError:
    print(f"ERROR: Could not find '{LIBRARY_FILE}'")
    libraries = []
except Exception as e:
    print(f"ERROR loading library data: {e}")
    libraries = []

# Read shapefile for Alachua County 2020 census tracts
print("\nReading Alachua County shapefile...")
try:
    geometries = read_shp(SHAPEFILE_SHP)
    records, fields = read_dbf(SHAPEFILE_DBF)
    print(f"✓ Found {len(records)} census tracts in Florida shapefile")
except FileNotFoundError as e:
    print(f"ERROR: Could not find shapefile: {e}")
    print("Please check that:")
    print("  1. Google Drive is mounted")
    print("  2. The .shp and .dbf files exist in your 'capstone' folder")
    print("  3. The filenames match exactly")
    geometries = []
    records = []

# Filter to Alachua County (COUNTYFP == '001' and STATEFP == '12')
alachua_features = []
fb_values = []

for i, record in enumerate(records):
    if record.get('STATEFP') == '12' and record.get('COUNTYFP') == '001':
        if geometries[i]:
            # Calculate square miles
            aland_sqmi = record.get('ALAND', 0) / 2589988.11  # Convert sq meters to sq miles
            awater_sqmi = record.get('AWATER', 0) / 2589988.11

            tractce_raw = record.get('TRACTCE', '')
            tractce_converted = convert_tractce_to_csv_format(tractce_raw)

            # Try to match with census data
            fb_count = None
            fb_pct = None
            total_pop = None
            if census_data is not None and tractce_converted:
                match = census_data[census_data['TRACTCE'] == tractce_converted]
                if not match.empty:
                    fb_count = int(match.iloc[0]['FB'])
                    fb_pct = float(match.iloc[0]['FB_pct'])
                    total_pop = int(match.iloc[0]['Total'])
                    fb_values.append(fb_count)

            alachua_features.append({
                "type": "Feature",
                "geometry": geometries[i],
                "properties": {
                    "GEOID": record.get('GEOID', ''),
                    "NAME": record.get('NAME', ''),
                    "TRACTCE": tractce_converted,  # Use converted format for display
                    "TRACTCE_RAW": tractce_raw,  # Keep original for reference
                    "ALAND": record.get('ALAND', 0),
                    "AWATER": record.get('AWATER', 0),
                    "ALAND_SQMI": round(aland_sqmi, 2),
                    "AWATER_SQMI": round(awater_sqmi, 3),
                    "FB": fb_count,
                    "FB_pct": fb_pct,
                    "Total": total_pop
                }
            })

print(f"✓ Filtered to {len(alachua_features)} Alachua County census tracts")
if census_data is not None:
    matched = sum(1 for f in alachua_features if f['properties']['FB'] is not None)
    print(f"✓ Matched {matched} tracts with FB data")
    # Show some example FB values
    fb_with_values = [f['properties'] for f in alachua_features if f['properties']['FB'] is not None]
    if fb_with_values:
        print(f"  Sample FB values:")
        for prop in sorted(fb_with_values, key=lambda x: x['FB'])[:3]:
            print(f"    Tract {prop['TRACTCE']}: FB = {prop['FB']}")
        print(f"    ...")
        for prop in sorted(fb_with_values, key=lambda x: x['FB'])[-3:]:
            print(f"    Tract {prop['TRACTCE']}: FB = {prop['FB']}")

geojson_alachua = {
    "type": "FeatureCollection",
    "features": alachua_features
}


# =========================
# 2. CREATE BASE MAP
# =========================

if stops:
    # Calculate center from stops
    lats = [s['Latitude'] for s in stops]
    lons = [s['Longitude'] for s in stops]
    center_lat = sum(lats) / len(lats)
    center_lon = sum(lons) / len(lons)
else:
    # Default to Gainesville center
    center_lat = 29.65163
    center_lon = -82.32483

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles="OpenStreetMap"
)


# =========================
# 3. CREATE COLOR SCALE FOR FB VALUES
# =========================

if fb_values:
    min_fb = min(fb_values)
    max_fb = max(fb_values)
    print(f"\nFB values range: {min_fb} to {max_fb}")
    print(f"Number of tracts with FB data: {len(fb_values)}")

    # Create colormap with more distinct steps (light blue to dark blue)
    colormap = cm.LinearColormap(
        colors=['#f0f9ff', '#bae6fd', '#7dd3fc', '#38bdf8', '#0ea5e9', '#0284c7', '#0369a1', '#075985', '#0c4a6e'],
        vmin=min_fb,
        vmax=max_fb,
        caption='Foreign Born Population (FB)'
    )

    def get_color(fb_value):
        """Get color for a given FB value"""
        if fb_value is None:
            return '#cccccc'  # Gray for no data
        # Make sure we're working with the actual numeric value
        color = colormap(fb_value)
        return color
else:
    def get_color(fb_value):
        return '#3388ff'


# =========================
# 4. ADD ALACHUA COUNTY CENSUS TRACTS (2020) WITH FB COLORING
# =========================

if alachua_features:
    folium.GeoJson(
        geojson_alachua,
        name="Alachua County Census Tracts (FB Population)",
        style_function=lambda feature: {
            "fillColor": get_color(feature['properties']['FB']),
            "color": "#000000",
            "weight": 1,
            "fillOpacity": 0.7,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["TRACTCE", "FB", "FB_pct", "Total", "ALAND_SQMI"],
            aliases=["Tract:", "Foreign Born:", "FB %:", "Total Pop:", "Land (sq mi):"],
            sticky=True
        ),
    ).add_to(m)

    # Add colormap legend
    if fb_values:
        colormap.add_to(m)

# =========================
# 5. ADD BUS ROUTES
# =========================

if routes:
    bus_routes_group = folium.FeatureGroup(name="Bus Routes", show=True)

    for route in routes:
        geom = route['geometry']
        route_num = route['route_shor']

        # Determine color
        color_hex = route.get('Colors', '')
        if color_hex and color_hex.strip():
            route_color = f"#{color_hex}"
        else:
            route_color = route_colors.get(route_num, '#3388ff')

        popup_html = f"""
        <div style="font-family: Arial; min-width: 250px;">
            <b style="font-size: 16px;">Route {route['route_shor']}</b><br>
            <b>{route['route_long']}</b><br>
            <hr style="margin: 5px 0;">
            <b>Shape ID:</b> {route['shape_id']}<br>
            <b>Hours:</b> {route['start_time']} - {route['end_time']}<br>
            <b>Length:</b> {route['length_mi']:.2f} miles<br>
            <hr style="margin: 5px 0;">
            <b>Frequency:</b><br>
            • AM: {route['AM_freq']:.0f} min<br>
            • Midday: {route['Midday_fre']:.0f} min<br>
            • PM: {route['PM_freq']:.0f} min<br>
        </div>
        """

        feature = {
            "type": "Feature",
            "geometry": geom,
            "properties": {
                "route": route_num,
                "name": route['route_long'],
                "shape_id": route['shape_id'],
                "color": route_color
            }
        }

        folium.GeoJson(
            feature,
            style_function=lambda x: {
                "color": x['properties']['color'],
                "weight": 4,
                "opacity": 0.8,
            },
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"Route {route_num}: {route['route_long']}"
        ).add_to(bus_routes_group)

    bus_routes_group.add_to(m)

    # Keep routes_by_number for the bus stop color logic below
    routes_by_number = {}
    for route in routes:
        route_num = route['route_shor']
        if route_num not in routes_by_number:
            routes_by_number[route_num] = []
        routes_by_number[route_num].append(route)

# =========================
# 6. ADD BUS STOP MARKERS
# =========================

if stops and routes:
    # First, determine which routes serve each stop (by proximity)
    # A stop is considered served by a route if it's within 50 meters of the route line

    def point_to_line_distance(point_lon, point_lat, line_coords):
        """Calculate minimum distance from point to line (in meters, approximate)"""
        min_dist = float('inf')

        for i in range(len(line_coords) - 1):
            x1, y1 = line_coords[i]
            x2, y2 = line_coords[i + 1]

            # Simple distance calculation (approximation)
            # Convert lat/lon difference to meters (rough estimate)
            dx = (point_lon - x1) * 111320 * math.cos(math.radians(point_lat))
            dy = (point_lat - y1) * 110540

            # Distance to segment
            segment_len_sq = ((x2 - x1) * 111320 * math.cos(math.radians(y1)))**2 + ((y2 - y1) * 110540)**2

            if segment_len_sq == 0:
                dist = math.sqrt(dx**2 + dy**2)
            else:
                # Project point onto line segment
                t = max(0, min(1, (dx * (x2 - x1) * 111320 * math.cos(math.radians(y1)) +
                                   dy * (y2 - y1) * 110540) / segment_len_sq))
                proj_x = x1 + t * (x2 - x1)
                proj_y = y1 + t * (y2 - y1)

                dx_proj = (point_lon - proj_x) * 111320 * math.cos(math.radians(point_lat))
                dy_proj = (point_lat - proj_y) * 110540
                dist = math.sqrt(dx_proj**2 + dy_proj**2)

            min_dist = min(min_dist, dist)

        return min_dist

    # Determine which routes serve each stop
    stop_routes = {}  # stop index -> list of route numbers
    proximity_threshold = 50  # meters

    for stop_idx, stop in enumerate(stops):
        serving_routes = set()

        for route in routes:
            geom = route['geometry']
            route_num = route['route_shor']

            if geom['type'] == 'LineString':
                dist = point_to_line_distance(stop['Longitude'], stop['Latitude'], geom['coordinates'])
                if dist < proximity_threshold:
                    serving_routes.add(route_num)
            elif geom['type'] == 'MultiLineString':
                for line in geom['coordinates']:
                    dist = point_to_line_distance(stop['Longitude'], stop['Latitude'], line)
                    if dist < proximity_threshold:
                        serving_routes.add(route_num)
                        break

        stop_routes[stop_idx] = sorted(list(serving_routes))

    # Get route colors from earlier definition
    # Multi-route color (for stops served by 2+ routes)
    multi_route_color = '#9333EA'  # Purple - not used by any route

    bus_feature_group = folium.FeatureGroup(name="Bus Stops")

    for stop_idx, stop in enumerate(stops):
        serving_routes = stop_routes.get(stop_idx, [])

        # Create detailed popup with amenities
        amenities = []
        if stop.get('NO_BENCHES', 0) > 0:
            amenities.append(f"🪑 {stop['NO_BENCHES']} bench(es)")
        if stop.get('NO_SHELTER', 0) > 0:
            amenities.append(f"🏠 {stop['NO_SHELTER']} shelter(s)")
        if stop.get('NO_BIKERAC', 0) > 0:
            amenities.append(f"🚲 {stop['NO_BIKERAC']} bike rack(s)")

        amenities_html = "<br>".join(amenities) if amenities else "No amenities"

        ada_status = stop.get('LANDING_PA', 'Unknown')
        ada_color = '🟢' if 'Compliant' in ada_status else '🔴' if 'Non-Compliant' in ada_status else '⚪'

        # Add routes information
        routes_html = ""
        if serving_routes:
            routes_html = f"<b>Routes:</b> {', '.join(serving_routes)}<br>"

        popup_html = f"""
        <div style="font-family: Arial; min-width: 200px;">
            <b style="font-size: 14px;">{stop['stop_name']}</b><br>
            <i>{stop['stop_desc']}</i><br>
            <hr style="margin: 5px 0;">
            {routes_html}
            <b>Street:</b> {stop.get('STREET', 'N/A')}<br>
            <b>Status:</b> {stop.get('ACTIVE_INA', 'Unknown')}<br>
            <b>ADA:</b> {ada_color} {ada_status}<br>
            <hr style="margin: 5px 0;">
            <b>Amenities:</b><br>
            {amenities_html}
        </div>
        """

        # Determine marker color based on routes
        if len(serving_routes) == 0:
            # No route nearby - gray
            marker_color = '#6B7280'
            fill_color = '#9CA3AF'
        elif len(serving_routes) == 1:
            # Single route - use that route's color
            route_num = serving_routes[0]
            color_hex = None

            # Find the color for this route
            for route in routes:
                if route['route_shor'] == route_num:
                    color_hex = route.get('Colors', '')
                    break

            if color_hex and color_hex.strip():
                marker_color = f"#{color_hex}"
                fill_color = f"#{color_hex}"
            else:
                marker_color = route_colors.get(route_num, '#3388ff')
                fill_color = route_colors.get(route_num, '#3388ff')
        else:
            # Multiple routes - use multi-route color (purple)
            marker_color = multi_route_color
            fill_color = multi_route_color

        # Adjust opacity for inactive stops
        if stop.get('ACTIVE_INA') != 'Active':
            opacity = 0.4
        else:
            opacity = 0.9

        folium.CircleMarker(
            location=[stop["Latitude"], stop["Longitude"]],
            radius=3,
            popup=folium.Popup(popup_html, max_width=300),
            color=marker_color,
            fill=True,
            fillColor=fill_color,
            fillOpacity=opacity,
            opacity=opacity,
            weight=2,
        ).add_to(bus_feature_group)

    bus_feature_group.add_to(m)

elif stops:
    # Fallback if routes aren't loaded
    bus_feature_group = folium.FeatureGroup(name="Bus Stops")

    for stop in stops:
        # Create detailed popup with amenities
        amenities = []
        if stop.get('NO_BENCHES', 0) > 0:
            amenities.append(f"🪑 {stop['NO_BENCHES']} bench(es)")
        if stop.get('NO_SHELTER', 0) > 0:
            amenities.append(f"🏠 {stop['NO_SHELTER']} shelter(s)")
        if stop.get('NO_BIKERAC', 0) > 0:
            amenities.append(f"🚲 {stop['NO_BIKERAC']} bike rack(s)")

        amenities_html = "<br>".join(amenities) if amenities else "No amenities"

        ada_status = stop.get('LANDING_PA', 'Unknown')
        ada_color = '🟢' if 'Compliant' in ada_status else '🔴' if 'Non-Compliant' in ada_status else '⚪'

        popup_html = f"""
        <div style="font-family: Arial; min-width: 200px;">
            <b style="font-size: 14px;">{stop['stop_name']}</b><br>
            <i>{stop['stop_desc']}</i><br>
            <hr style="margin: 5px 0;">
            <b>Street:</b> {stop.get('STREET', 'N/A')}<br>
            <b>Status:</b> {stop.get('ACTIVE_INA', 'Unknown')}<br>
            <b>ADA:</b> {ada_color} {ada_status}<br>
            <hr style="margin: 5px 0;">
            <b>Amenities:</b><br>
            {amenities_html}
        </div>
        """

        # Color code by active status
        if stop.get('ACTIVE_INA') == 'Active':
            marker_color = 'green'
            fill_color = 'lightgreen'
        else:
            marker_color = 'red'
            fill_color = 'lightcoral'

        folium.CircleMarker(
            location=[stop["Latitude"], stop["Longitude"]],
            radius=3,
            popup=folium.Popup(popup_html, max_width=300),
            color=marker_color,
            fill=True,
            fillColor=fill_color,
            fill_opacity=0.8,
        ).add_to(bus_feature_group)

    bus_feature_group.add_to(m)


# =========================
# 7. ADD LIBRARY LOCATIONS
# =========================

if libraries:
    library_feature_group = folium.FeatureGroup(name="Libraries")

    for library in libraries:
        # Create popup with library name
        popup_html = f"""
        <div style="font-family: Arial; min-width: 150px;">
            <b style="font-size: 14px;">📚 {library['Branch']}</b><br>
            <i>Alachua County Library</i>
        </div>
        """

        # Use Google Maps style marker (red pin)
        folium.Marker(
            location=[library["Latitude"], library["Longitude"]],
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=library['Branch'],
            icon=folium.Icon(color='red', icon='book', prefix='fa')
        ).add_to(library_feature_group)

    library_feature_group.add_to(m)


# =========================
# 8. ADD GAINESVILLE CITY BOUNDARY (TOP LAYER)
# =========================

if gainesville_geom:
    city_geojson = {
        "type": "Feature",
        "geometry": gainesville_geom,
        "properties": {
            "name": "Gainesville",
            "acres": gainesville_acres if gainesville_acres else "N/A",
            "sq_mi": f"{gainesville_acres/640:.1f}" if gainesville_acres else "N/A"
        }
    }

    folium.GeoJson(
        city_geojson,
        name="Gainesville City Limits (2021)",
        style_function=lambda feature: {
            "fillColor": "none",
            "color": "#ff0000",  # Red boundary
            "weight": 3,
            "fillOpacity": 0,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["name", "sq_mi"],
            aliases=["City:", "Area (sq mi):"],
            sticky=True
        ),
    ).add_to(m)


# =========================
# 9. LAYER CONTROLS
# =========================

folium.LayerControl(collapsed=False).add_to(m)


# =========================
# 10. ADD TITLE, COMPASS ROSE AND SCALE
# =========================

# Add a title to the map
title_html = '''
<div style="position: fixed;
            top: 10px;
            left: 50%;
            transform: translateX(-50%);
            width: auto;
            height: auto;
            z-index: 9999;
            background-color: rgba(255, 255, 255, 0.9);
            border: 2px solid #2c3e50;
            border-radius: 8px;
            padding: 10px 20px;
            box-shadow: 0 3px 8px rgba(0,0,0,0.4);
            font-family: Cambria, sans-serif;
            font-size: 20px;
            font-weight: bold;
            color: #2c3e50;
            text-align: center;">
    Comprehensive Gainesville Map
</div>
'''

m.get_root().html.add_child(folium.Element(title_html))

# Add a compass rose using HTML/CSS (fixed position, stays same size)
compass_html = '''
<div style="position: fixed;
            top: 100px;
            left: 15px;
            width: 70px;
            height: 70px;
            z-index: 9999;
            background: radial-gradient(circle, rgba(255,255,255,0.95) 0%, rgba(240,240,240,0.9) 100%);
            border: 2.5px solid #2c3e50;
            border-radius: 50%;
            box-shadow: 0 3px 8px rgba(0,0,0,0.4);
            display: flex;
            align-items: center;
            justify-content: center;
            font-family: 'Arial', sans-serif;">
    <div style="position: relative; width: 100%; height: 100%;">
        <!-- North pointer (red) -->
        <div style="position: absolute;
                    top: 50%;
                    left: 50%;
                    width: 0;
                    height: 0;
                    border-left: 6px solid transparent;
                    border-right: 6px solid transparent;
                    border-bottom: 24px solid #c0392b;
                    transform: translate(-50%, -100%) translateY(6px);
                    filter: drop-shadow(0 1px 2px rgba(0,0,0,0.3));"></div>

        <!-- South pointer (gray) -->
        <div style="position: absolute;
                    top: 50%;
                    left: 50%;
                    width: 0;
                    height: 0;
                    border-left: 6px solid transparent;
                    border-right: 6px solid transparent;
                    border-top: 24px solid #7f8c8d;
                    transform: translate(-50%, 0%) translateY(-6px);
                    filter: drop-shadow(0 1px 2px rgba(0,0,0,0.2));"></div>

        <!-- East pointer -->
        <div style="position: absolute;
                    top: 50%;
                    left: 50%;
                    width: 0;
                    height: 0;
                    border-top: 5px solid transparent;
                    border-bottom: 5px solid transparent;
                    border-left: 20px solid #95a5a6;
                    transform: translate(0%, -50%) translateX(-6px);"></div>

        <!-- West pointer -->
        <div style="position: absolute;
                    top: 50%;
                    left: 50%;
                    width: 0;
                    height: 0;
                    border-top: 5px solid transparent;
                    border-bottom: 5px solid transparent;
                    border-right: 20px solid #95a5a6;
                    transform: translate(-100%, -50%) translateX(6px);"></div>

        <!-- Center circle -->
        <div style="position: absolute;
                    top: 50%;
                    left: 50%;
                    width: 8px;
                    height: 8px;
                    background-color: #34495e;
                    border-radius: 50%;
                    transform: translate(-50%, -50%);
                    box-shadow: 0 0 3px rgba(0,0,0,0.5);"></div>

        <!-- Cardinal letters -->
        <div style="position: absolute; top: 3px; left: 50%; transform: translateX(-50%);
                    color: #c0392b; font-weight: bold; font-size: 13px; text-shadow: 0 1px 2px rgba(255,255,255,0.8);">N</div>
        <div style="position: absolute; bottom: 3px; left: 50%; transform: translateX(-50%);
                    color: #7f8c8d; font-weight: bold; font-size: 11px;">S</div>
        <div style="position: absolute; right: 3px; top: 50%; transform: translateY(-50%);
                    color: #95a5a6; font-weight: bold; font-size: 11px;">E</div>
        <div style="position: absolute; left: 3px; top: 50%; transform: translateY(-50%);
                    color: #95a5a6; font-weight: bold; font-size: 11px;">W</div>
    </div>
</div>
'''

m.get_root().html.add_child(folium.Element(compass_html))

# Add measure control (includes a scale bar and distance measurement tool)
from folium.plugins import MeasureControl
m.add_child(MeasureControl(
    position='bottomleft',
    primary_length_unit='miles',
    secondary_length_unit='meters',
    primary_area_unit='sqmiles',
    secondary_area_unit='sqmeters'
))


# =========================
# 11. SAVE MAP
# =========================

# Save to Google Drive
output_path = "/content/drive/MyDrive/capstone/gainesville_bus_map_complete.html"
m.save(output_path)

print("\n" + "="*50)
print(f"Map saved to: {output_path}")
print("="*50)
print(f"Alachua County 2020 Census Tracts: {len(alachua_features)}")
if routes:
    print(f"Bus Routes: {len(routes)} route segments")
    print(f"  Unique routes: {len(routes_by_number)}")
if stops:
    print(f"Bus Stops: {len(stops)}")
    active_stops = sum(1 for s in stops if s.get('ACTIVE_INA') == 'Active')
    print(f"  - Active: {active_stops}")
    print(f"  - Inactive: {len(stops) - active_stops}")
    if routes:
        multi_route_stops = sum(1 for idx in stop_routes if len(stop_routes[idx]) >= 2)
        print(f"  - Multi-route stops: {multi_route_stops}")
        print(f"  - Single-route stops: {len(stops) - multi_route_stops - sum(1 for idx in stop_routes if len(stop_routes[idx]) == 0)}")
        print(f"  - No nearby route: {sum(1 for idx in stop_routes if len(stop_routes[idx]) == 0)}")
if gainesville_geom:
    print(f"Gainesville City Limits: {gainesville_acres/640:.1f} sq mi")
if libraries:
    print(f"Libraries: {len(libraries)}")
print("\nLayers included:")
if alachua_features:
    print("  ✓ Alachua County Census Tracts (2020) - Colored by FB population")
    print("    (Light blue = low FB, Dark blue = high FB)")
if gainesville_geom:
    print("  ✓ Gainesville City Limits (2021) - Red boundary")
if routes:
    print("  ✓ Bus Routes - Color-coded by route number")
    print("    (Each route can be toggled on/off individually)")
if stops:
    if routes:
        print("  ✓ Bus Stops - Color-coded by routes served")
        print("    (Single route = route color, 2+ routes = purple, none = gray)")
    else:
        print("  ✓ Bus Stops - Green markers")
if libraries:
    print("  ✓ Libraries - Red book markers")
print("\nYou can toggle layers on/off using the layer control!")
print("Hover over census tracts and routes to see data!")

Loading data...
✓ Loaded 974 bus stops from shapefile
✓ Loaded 52 bus routes from shapefile
  Unique routes: ['1', '10', '11', '118', '12', '13', '15', '17', '20', '21', '23', '26', '3', '33', '37', '38', '43', '5', '52', '55', '6', '7', '75', '76', '8', '9']
✓ Loaded 58 census tracts with FB data
✓ Loaded Gainesville city limits (41511.8 acres = 64.9 sq mi)
✓ Loaded 12 library locations

Reading Alachua County shapefile...
✓ Found 5160 census tracts in Florida shapefile
✓ Filtered to 58 Alachua County census tracts
✓ Matched 58 tracts with FB data
  Sample FB values:
    Tract 6: FB = 19
    Tract 20.01: FB = 48
    Tract 21.02: FB = 49
    ...
    Tract 15.22: FB = 1444
    Tract 16.05: FB = 1727
    Tract 22.04: FB = 1907

FB values range: 19 to 1907
Number of tracts with FB data: 58

Map saved to: /content/drive/MyDrive/capstone/gainesville_bus_map_complete.html
Alachua County 2020 Census Tracts: 58
Bus Routes: 52 route segments
  Unique routes: 26
Bus Stops: 974
  - Active: 974
  